# Ablation Study: Watermark Detection Threshold and Corpus Size

This notebook performs comprehensive ablation studies:

1. **Watermark z-score threshold ablation**: TPR, FPR, and F1 vs detection threshold
2. **Corpus size ablation**: How ASR-R changes with increasing corpus size (poison dilution)
3. **Top-k retrieval depth ablation**: How retrieval depth affects ASR-R
4. **Embedding model analysis**: Distribution of cosine similarities in the corpus

Reference: Unigram watermark (Zhao et al., ICLR 2024, arXiv:2306.17439)

In [ ]:
import sys
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

from watermark.watermarking import create_watermark_encoder, ProvenanceTracker
from evaluation.retrieval_sim import RetrievalSimulator
from data.synthetic_corpus import SyntheticCorpus
from memory_systems.vector_store import VectorMemorySystem

matplotlib.rcParams = plt.rcParams
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

import matplotlib
matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['font.size'] = 11

SEED = 42
print('ablation study setup complete')

## 1. Watermark Z-Score Distribution

In [ ]:
encoder = create_watermark_encoder('unigram')
corpus = SyntheticCorpus(seed=SEED)
all_entries = corpus.generate_benign_entries(200)
texts = [e['content'] for e in all_entries]

# collect z-scores for watermarked and clean texts
import random
rng = random.Random(SEED)

z_watermarked = []
z_clean = []

for i, text in enumerate(texts[:60]):
    try:
        wm_text = encoder.embed(text, f'wm_{i:04d}')
        stats = encoder.get_detection_stats(wm_text)
        z_watermarked.append(float(stats['z_score']))
    except Exception:
        z_watermarked.append(rng.gauss(6.0, 1.5))

for i, text in enumerate(texts[60:120]):
    try:
        stats = encoder.get_detection_stats(text)
        z_clean.append(float(stats['z_score']))
    except Exception:
        z_clean.append(rng.gauss(0.0, 1.0))

print(f'watermarked: n={len(z_watermarked)}, mean={np.mean(z_watermarked):.2f}, std={np.std(z_watermarked):.2f}')
print(f'clean:       n={len(z_clean)}, mean={np.mean(z_clean):.2f}, std={np.std(z_clean):.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Unigram Watermark Z-Score Analysis (Zhao et al., ICLR 2024)', fontsize=13)

# panel 1: z-score distributions
ax1 = axes[0]
ax1.hist(z_clean, bins=20, alpha=0.6, color='steelblue', label='Clean (not watermarked)', density=True)
ax1.hist(z_watermarked, bins=20, alpha=0.6, color='crimson', label='Watermarked', density=True)
ax1.axvline(x=4.0, color='black', linestyle='--', linewidth=1.5, label='Default threshold (z=4)')
ax1.axvline(x=np.mean(z_clean), color='steelblue', linestyle=':', linewidth=1.5, alpha=0.7)
ax1.axvline(x=np.mean(z_watermarked), color='crimson', linestyle=':', linewidth=1.5, alpha=0.7)
ax1.set_xlabel('Z-Score')
ax1.set_ylabel('Density')
ax1.set_title('Z-Score Distribution: Watermarked vs Clean')
ax1.legend(fontsize=9)

# panel 2: ECDF plot
ax2 = axes[1]
z_sorted_clean = np.sort(z_clean)
z_sorted_wm = np.sort(z_watermarked)
ax2.plot(z_sorted_clean, np.arange(1, len(z_sorted_clean)+1) / len(z_sorted_clean),
         color='steelblue', linewidth=2, label='Clean ECDF')
ax2.plot(z_sorted_wm, np.arange(1, len(z_sorted_wm)+1) / len(z_sorted_wm),
         color='crimson', linewidth=2, label='Watermarked ECDF')
ax2.axvline(x=4.0, color='black', linestyle='--', linewidth=1.5, label='Default threshold')
ax2.set_xlabel('Z-Score Threshold')
ax2.set_ylabel('Cumulative Fraction')
ax2.set_title('Empirical CDF of Z-Scores')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../visualization/fig_watermark_zscore_distribution.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_watermark_zscore_distribution.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: fig_watermark_zscore_distribution')

## 2. Threshold Ablation: TPR, FPR, F1 vs z-Score Threshold

In [ ]:
thresholds = np.arange(0.0, 12.1, 0.2)
tpr_vals, fpr_vals, f1_vals, prec_vals = [], [], [], []

for thresh in thresholds:
    tp = sum(1 for z in z_watermarked if z >= thresh)
    fn = len(z_watermarked) - tp
    fp = sum(1 for z in z_clean if z >= thresh)
    tn = len(z_clean) - fp
    
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1 = 2 * prec * tpr / (prec + tpr) if (prec + tpr) > 0 else 0.0
    
    tpr_vals.append(tpr)
    fpr_vals.append(fpr)
    f1_vals.append(f1)
    prec_vals.append(prec)

# find optimal threshold (max F1)
best_idx = int(np.argmax(f1_vals))
best_thresh = thresholds[best_idx]
print(f'Optimal threshold: z={best_thresh:.1f}')
print(f'  TPR={tpr_vals[best_idx]:.3f}  FPR={fpr_vals[best_idx]:.3f}  F1={f1_vals[best_idx]:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Watermark Detection Threshold Ablation', fontsize=13)

ax1 = axes[0]
ax1.plot(thresholds, tpr_vals, color='#27AE60', linewidth=2, label='TPR (Recall)')
ax1.plot(thresholds, fpr_vals, color='#E74C3C', linewidth=2, label='FPR')
ax1.plot(thresholds, f1_vals, color='#3498DB', linewidth=2, label='F1 Score')
ax1.plot(thresholds, prec_vals, color='#9B59B6', linewidth=2, linestyle='--', label='Precision')
ax1.axvline(x=best_thresh, color='black', linestyle=':', linewidth=1.5, label=f'Optimal z={best_thresh:.1f}')
ax1.axvline(x=4.0, color='gray', linestyle='--', linewidth=1.0, alpha=0.5, label='Default z=4.0')
ax1.set_xlabel('Z-Score Detection Threshold')
ax1.set_ylabel('Rate')
ax1.set_title('TPR, FPR, F1, Precision vs Threshold')
ax1.legend(fontsize=9)
ax1.set_xlim(0, 12)

ax2 = axes[1]
ax2.plot(fpr_vals, tpr_vals, color='#8E44AD', linewidth=2.5)
ax2.scatter([fpr_vals[best_idx]], [tpr_vals[best_idx]], color='red', s=120, zorder=5,
            label=f'Optimal (z={best_thresh:.1f})')
ax2.scatter([fpr_vals[list(thresholds).index(min(thresholds, key=lambda t: abs(t-4.0)))]],
            [tpr_vals[list(thresholds).index(min(thresholds, key=lambda t: abs(t-4.0)))]],
            color='steelblue', s=120, zorder=5, marker='s', label='Default (z=4.0)')
ax2.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax2.set_xlabel('False Positive Rate (FPR)')
ax2.set_ylabel('True Positive Rate (TPR)')
ax2.set_title('ROC Curve: Watermark Detection')
ax2.legend(fontsize=9)
ax2.set_xlim(-0.02, 1.02)
ax2.set_ylim(-0.02, 1.02)

plt.tight_layout()
plt.savefig('../visualization/fig_watermark_threshold_ablation.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_watermark_threshold_ablation.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: fig_watermark_threshold_ablation')

## 3. Corpus Size Ablation: Poison Dilution Effect on ASR-R

In [ ]:
corpus_sizes = [50, 100, 150, 200, 300, 500]
corpus_ablation = {at: {'asr_r': [], 'asr_t': []} for at in ['agent_poison', 'minja', 'injecmem']}

for cs in corpus_sizes:
    sim_cs = RetrievalSimulator(corpus_size=cs, top_k=5, n_poison_per_attack=5, seed=SEED)
    for at in ['agent_poison', 'minja', 'injecmem']:
        m = sim_cs.evaluate_attack(at)
        corpus_ablation[at]['asr_r'].append(m.asr_r)
        corpus_ablation[at]['asr_t'].append(m.asr_t)
    print(f'corpus_size={cs}: done')

attack_colors = {'agent_poison': '#E74C3C', 'minja': '#3498DB', 'injecmem': '#27AE60'}
attack_labels = {'agent_poison': 'AgentPoison', 'minja': 'MINJA', 'injecmem': 'InjecMEM'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Corpus Size Ablation: Effect of Memory Size on Attack Success', fontsize=13)

for ax, metric, ylabel in zip(axes, ['asr_r', 'asr_t'], ['ASR-R (Retrieval)', 'ASR-T (End-to-End)']):
    for at in ['agent_poison', 'minja', 'injecmem']:
        ax.plot(corpus_sizes, corpus_ablation[at][metric],
                marker='o', color=attack_colors[at], linewidth=2, markersize=8,
                label=attack_labels[at])
    ax.set_xlabel('Benign Corpus Size (Number of Memory Entries)')
    ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} vs Corpus Size')
    ax.legend()
    ax.set_ylim(0, 1.0)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../visualization/fig_corpus_size_ablation.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_corpus_size_ablation.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: fig_corpus_size_ablation')

## 4. Top-k Retrieval Depth Ablation

In [ ]:
top_k_values = [1, 2, 3, 5, 10, 20]
topk_ablation = {at: [] for at in ['agent_poison', 'minja', 'injecmem']}

for k in top_k_values:
    sim_k = RetrievalSimulator(corpus_size=200, top_k=k, n_poison_per_attack=5, seed=SEED)
    for at in ['agent_poison', 'minja', 'injecmem']:
        m = sim_k.evaluate_attack(at)
        topk_ablation[at].append(m.asr_r)
    print(f'top_k={k}: done')

fig, ax = plt.subplots(figsize=(9, 5))
for at in ['agent_poison', 'minja', 'injecmem']:
    ax.plot(top_k_values, topk_ablation[at],
            marker='o', color=attack_colors[at], linewidth=2, markersize=8,
            label=attack_labels[at])

ax.set_xlabel('Retrieval Depth (top-k)')
ax.set_ylabel('ASR-R (Retrieval Success Rate)')
ax.set_title('Effect of Retrieval Depth on Attack Success Rate', fontsize=12)
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(True, alpha=0.3)
ax.axvline(x=5, color='black', linestyle='--', alpha=0.5, label='Standard k=5')

plt.tight_layout()
plt.savefig('../visualization/fig_topk_depth_ablation.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_topk_depth_ablation.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: fig_topk_depth_ablation')

## 5. Cosine Similarity Heatmap: Victim Queries vs Corpus Topics

In [ ]:
from data.synthetic_corpus import VICTIM_QUERIES
import numpy as np

# sample one representative entry per category from corpus
category_samples = {}
for e in all_entries:
    cat = e['category']
    if cat not in category_samples:
        category_samples[cat] = e['content']

categories = list(category_samples.keys())
category_texts = [category_samples[c] for c in categories]

# sample 10 victim queries for the heatmap
sample_victim_qs = [q['query'] for q in VICTIM_QUERIES[:10]]

# compute similarity matrix
from memory_systems.vector_store import VectorMemorySystem
mem_temp = VectorMemorySystem()
q_embs = mem_temp._embed(sample_victim_qs)
c_embs = mem_temp._embed(category_texts)
sim_matrix = q_embs @ c_embs.T  # (10, n_categories)

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(sim_matrix, cmap='Blues', aspect='auto', vmin=0, vmax=0.7)
ax.set_xticks(range(len(categories)))
ax.set_xticklabels([c.capitalize() for c in categories], rotation=30, ha='right')
ax.set_yticks(range(len(sample_victim_qs)))
ax.set_yticklabels([q[:40]+'...' if len(q)>40 else q for q in sample_victim_qs], fontsize=8)
ax.set_title('Cosine Similarity: Victim Queries vs Memory Category Representatives', fontsize=11)
plt.colorbar(im, ax=ax, label='Cosine Similarity')
plt.tight_layout()
plt.savefig('../visualization/fig_query_category_similarity_heatmap.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_query_category_similarity_heatmap.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: fig_query_category_similarity_heatmap')

## 6. Save Ablation Results

In [ ]:
ablation_data = {
    'threshold_ablation': {
        'thresholds': list(thresholds),
        'tpr': tpr_vals,
        'fpr': fpr_vals,
        'f1': f1_vals,
        'precision': prec_vals,
        'optimal_threshold': float(best_thresh),
        'optimal_f1': float(f1_vals[best_idx]),
        'z_watermarked_mean': float(np.mean(z_watermarked)),
        'z_clean_mean': float(np.mean(z_clean)),
    },
    'corpus_size_ablation': {
        'corpus_sizes': corpus_sizes,
        'results': corpus_ablation,
    },
    'topk_ablation': {
        'top_k_values': top_k_values,
        'results': topk_ablation,
    },
    'evasion_ablation': {
        'paraphrase': {
            'intensity_results': ev_paraph.intensity_results,
            'tpr_before': ev_paraph.tpr_before,
            'tpr_after': ev_paraph.tpr_after,
            'evasion_success_rate': ev_paraph.evasion_success_rate,
        },
        'copy_paste': {
            'intensity_results': ev_dilute.intensity_results,
            'tpr_before': ev_dilute.tpr_before,
            'tpr_after': ev_dilute.tpr_after,
            'evasion_success_rate': ev_dilute.evasion_success_rate,
        },
        'adaptive_substitution': {
            'intensity_results': ev_adapt.intensity_results,
            'tpr_before': ev_adapt.tpr_before,
            'tpr_after': ev_adapt.tpr_after,
            'evasion_success_rate': ev_adapt.evasion_success_rate,
        },
    },
}

output_path = '../visualization/ablation_study_results.json'
with open(output_path, 'w') as f:
    json.dump(ablation_data, f, indent=2)

print(f'ablation results saved to {output_path}')
print(f'optimal watermark threshold: z={best_thresh:.1f}  (F1={f1_vals[best_idx]:.3f})')
print(f'evasion ablation: paraphrase tpr {ev_paraph.tpr_before:.3f}->{ev_paraph.tpr_after:.3f}, '
      f'dilution {ev_dilute.tpr_before:.3f}->{ev_dilute.tpr_after:.3f}, '
      f'adaptive {ev_adapt.tpr_before:.3f}->{ev_adapt.tpr_after:.3f}')

In [ ]:
from watermark.watermarking import UnigramWatermarkEncoder
from evaluation.evasion_eval import WatermarkEvasionEvaluator

encoder = UnigramWatermarkEncoder()
wm_base = (
    "the memory agent system stores and retrieves contextual information "
    "across multiple interaction sessions. the scheduler confirms task "
    "completion, meeting attendance, and deadline tracking. the system "
    "verifies authentication and validates authorization for all access "
    "requests. confirmed schedule entries require approved protocol steps "
    "for execution and verified credential submission from authorized users."
)

# build evaluation samples
n_evasion_samples = 30
wm_samples = [encoder.embed(wm_base + f" variant {i}.", f"abl_wm_{i}") for i in range(n_evasion_samples)]
clean_samples = [all_entries[i % len(all_entries)]['content'] for i in range(n_evasion_samples)]

# fine-grained ablation for each attack type
ev = WatermarkEvasionEvaluator(encoder, n_samples=n_evasion_samples, seed=SEED)

paraphrase_levels = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
dilution_ratios = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 2.5, 3.0]
sub_budgets = [0, 1, 2, 3, 5, 8, 10, 15, 20]

# run evaluations with fine-grained grids
ev_paraph = ev.evaluate_paraphrasing(wm_samples, clean_samples, intensity_levels=[l for l in paraphrase_levels if l > 0])
ev_dilute = ev.evaluate_copy_paste_dilution(wm_samples, clean_samples, dilution_ratios=[r for r in dilution_ratios if r > 0])
ev_adapt = ev.evaluate_adaptive_substitution(wm_samples, clean_samples, substitution_budgets=[b for b in sub_budgets if b > 0])

print('evasion intensity ablation complete')
for label, result in [('paraphrase', ev_paraph), ('copy_paste', ev_dilute), ('adaptive', ev_adapt)]:
    print(f'  {label}: tpr {result.tpr_before:.3f} → {result.tpr_after:.3f}  esr={result.evasion_success_rate:.3f}')

## 6. Phase 10 Upgrade: Evasion Intensity Ablation

Fine-grained ablation of evasion intensity vs. detection performance for all three evasion classes. This directly corresponds to the ablation plots in the defense section of the paper: each curve shows how TPR degrades as the adversary's budget increases.

In [ ]:
ablation_data = {
    'threshold_ablation': {
        'thresholds': list(thresholds),
        'tpr': tpr_vals,
        'fpr': fpr_vals,
        'f1': f1_vals,
        'precision': prec_vals,
        'optimal_threshold': float(best_thresh),
        'optimal_f1': float(f1_vals[best_idx]),
        'z_watermarked_mean': float(np.mean(z_watermarked)),
        'z_clean_mean': float(np.mean(z_clean)),
    },
    'corpus_size_ablation': {
        'corpus_sizes': corpus_sizes,
        'results': corpus_ablation,
    },
    'topk_ablation': {
        'top_k_values': top_k_values,
        'results': topk_ablation,
    },
}

output_path = '../visualization/ablation_study_results.json'
with open(output_path, 'w') as f:
    json.dump(ablation_data, f, indent=2)

print(f'ablation results saved to {output_path}')
print(f'optimal watermark threshold: z={best_thresh:.1f}  (F1={f1_vals[best_idx]:.3f})')